# Dog vs Cat Detector (YOLOv8)
Run the cells in order. **Before starting:** Runtime → Change runtime type → **T4 GPU**.

Dataset layout on Drive:
```
dog_cat_dataset/dog_cat/
├── train/images
└── test/images
```
Labels are created automatically in Step 5 (pseudo-labeling with YOLOv8x). Class comes from the filename (`cat...` / `dog...`).

## 1. Install

In [ ]:
!pip install -q ultralytics opencv-python matplotlib

## 2. Imports

In [ ]:
from ultralytics import YOLO
import cv2, os, glob, shutil
import matplotlib.pyplot as plt
import torch

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Check GPU and dataset folders

In [ ]:
print("GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Go to Runtime > Change runtime type > T4 GPU, then re-run from Step 2."
print(torch.cuda.get_device_name(0))

BASE = "/content/drive/MyDrive/dog_cat_dataset/dog_cat"   # your dataset on Drive
RUNS = "/content/drive/MyDrive/runs"                      # training results are saved here
print(os.listdir(BASE))
for s in ["train", "test"]:
    print(s, "images:", len(glob.glob(f"{BASE}/{s}/images/*")))

GPU available: True
Tesla T4
['train', 'test', 'data.yaml']
train images: 1001
test images: 0


## 5. Create labels (pseudo-labeling)
Skips images that already have a label file, so it is safe to re-run. Takes a few minutes on GPU.

In [ ]:
det = YOLO("yolov8x.pt")      # large COCO model: cat = 15, dog = 16

def label_split(split):
    root = f"{BASE}/{split}"
    os.makedirs(f"{root}/labels", exist_ok=True)
    made = skipped = nodet = 0
    files = [p for p in glob.glob(f"{root}/images/*")
             if p.lower().endswith((".jpg", ".jpeg", ".png"))]
    for p in files:
        stem = os.path.splitext(os.path.basename(p))[0]
        out = f"{root}/labels/{stem}.txt"
        if os.path.exists(out):
            skipped += 1
            continue
        cls = 0 if "cat" in os.path.basename(p).lower() else 1
        r = det(p, classes=[15, 16], verbose=False, device=0)[0]
        if len(r.boxes) == 0:
            nodet += 1
            continue
        b = r.boxes[r.boxes.conf.argmax()]          # most confident box
        x, y, w, h = b.xywhn[0].tolist()
        with open(out, "w") as f:
            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")
        made += 1
    print(f"{split}: created {made}, already existed {skipped}, no detection {nodet}")

label_split("train")
label_split("test")

train: created 0, already existed 991, no detection 9
test: created 0, already existed 0, no detection 0


## 6. Verify labels

In [ ]:
for s in ["train", "test"]:
    n_img = len(glob.glob(f"{BASE}/{s}/images/*"))
    n_lbl = len(glob.glob(f"{BASE}/{s}/labels/*.txt"))
    print(f"{s}: {n_img} images, {n_lbl} labels")
# Numbers should be close. A few missing labels are fine.

train: 1001 images, 991 labels
test: 0 images, 0 labels


## 7. Copy data to local disk and write data.yaml
Reading from Drive is slow, so training uses a local copy.

In [ ]:
LOCAL = "/content/dog_cat"
if os.path.exists(LOCAL):
    shutil.rmtree(LOCAL)
shutil.copytree(BASE, LOCAL, ignore=shutil.ignore_patterns("*.cache", "data.yaml"))

yaml_text = f"""path: {LOCAL}
train: train/images
val: test/images
names:
  0: cat
  1: dog
"""
with open(f"{LOCAL}/data.yaml", "w") as f:
    f.write(yaml_text)
print(yaml_text)

path: /content/dog_cat
train: train/images
val: test/images
names:
  0: cat
  1: dog



In [ ]:
for s in ["train", "test"]:
    for root, dirs, files in os.walk(f"{BASE}/{s}"):
        print(root, "| files:", len(files))

/content/drive/MyDrive/dog_cat_dataset/dog_cat/train | files: 1
/content/drive/MyDrive/dog_cat_dataset/dog_cat/train/images | files: 1001
/content/drive/MyDrive/dog_cat_dataset/dog_cat/train/labels | files: 991
/content/drive/MyDrive/dog_cat_dataset/dog_cat/test | files: 0
/content/drive/MyDrive/dog_cat_dataset/dog_cat/test/images | files: 0
/content/drive/MyDrive/dog_cat_dataset/dog_cat/test/labels | files: 0


In [ ]:
TEST_DIR = f"{LOCAL}/valid/images"

In [ ]:
import random

LOCAL = "/content/dog_cat"

# start clean: drop any partial valid folder and stale caches
shutil.rmtree(f"{LOCAL}/valid", ignore_errors=True)
for c in glob.glob(f"{LOCAL}/**/*.cache", recursive=True):
    os.remove(c)

random.seed(42)
imgs = sorted(glob.glob(f"{LOCAL}/train/images/*"))
val_imgs = random.sample(imgs, int(0.15 * len(imgs)))

os.makedirs(f"{LOCAL}/valid/images", exist_ok=True)
os.makedirs(f"{LOCAL}/valid/labels", exist_ok=True)

for p in val_imgs:
    stem = os.path.splitext(os.path.basename(p))[0]
    shutil.move(p, f"{LOCAL}/valid/images/")
    lbl = f"{LOCAL}/train/labels/{stem}.txt"
    if os.path.exists(lbl):
        shutil.move(lbl, f"{LOCAL}/valid/labels/")

with open(f"{LOCAL}/data.yaml", "w") as f:
    f.write(f"""path: {LOCAL}
train: train/images
val: valid/images
names:
  0: cat
  1: dog
""")

print(open(f"{LOCAL}/data.yaml").read())
print("train:", len(glob.glob(f"{LOCAL}/train/images/*")),
      "| valid:", len(glob.glob(f"{LOCAL}/valid/images/*")))

path: /content/dog_cat
train: train/images
val: valid/images
names:
  0: cat
  1: dog

train: 723 | valid: 127


## 8. Train

In [ ]:
model = YOLO("yolov8s.pt")
model.train(
    data=f"{LOCAL}/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,
    project=RUNS,
    name="dog_cat_detector",
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dog_cat/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dog_cat_detector-4, nbs=64, nms=None, opset=None, optimize=Fa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f657b237a10>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

## 9. Evaluate

In [ ]:
import glob, os, cv2
import matplotlib.pyplot as plt

SHOW_DIR = "/content/dog_cat/valid/images"   # images the model did NOT train on
MAX_IMAGES = 10
CONF = 0.4

files = sorted(glob.glob(f"{SHOW_DIR}/*"))[:MAX_IMAGES]
print("images found:", len(files))

for p in files:
    image = cv2.imread(p)
    if image is None:
        continue
    results = trained_model.predict(image, conf=CONF, verbose=False)
    annotated = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(8, 6))
    plt.imshow(annotated)
    plt.title(os.path.basename(p))
    plt.axis("off")
    plt.show()

Output hidden; open in https://colab.research.google.com to view.